# SpanCall — Train / Eval / Test Notebook

This notebook runs the **full SpanCall workflow**: **git clone/pull → env + HF dataset pull → tensorize → staged training (T0/T1/T2/C) → prediction → calibration → offline eval → test splits**.

> The public code repository (`https://github.com/eleluong/spancall.git`) is cloned or pulled from Git, dependencies are installed, and canonical dataset splits are pulled automatically from the Hugging Face Hub (`DATASET_NAME` in `.env`, default `convoicon/spancall`) when not already present in `DATA_DIR`.


---
## 0. Setup — Pull Code from Git & Install

Clones the public SpanCall repository (`https://github.com/eleluong/spancall.git`) or pulls the latest updates from `origin/main` if already cloned (e.g. running in Google Colab), enters the repository directory, adds it to `sys.path`, and installs dependencies in editable mode.

In [1]:
# 0a. Pull latest code from public GitHub repository and install dependencies
import os, sys
from pathlib import Path

GIT_REPO_URL = "https://github.com/eleluong/spancall.git"
GIT_BRANCH   = "main"

# Locate or clone repository
cwd = Path.cwd().resolve()
if (cwd / "spancall" / "__init__.py").is_file():
    PROJECT_ROOT = cwd
elif (cwd.parent / "spancall" / "__init__.py").is_file():
    PROJECT_ROOT = cwd.parent
elif (cwd / "spancall" / "spancall" / "__init__.py").is_file():
    PROJECT_ROOT = cwd / "spancall"
else:
    print(f"Cloning public repository from {GIT_REPO_URL}...")
    !git clone --branch {GIT_BRANCH} {GIT_REPO_URL}
    PROJECT_ROOT = cwd / "spancall"

# Change directory to project root and ensure it is on sys.path
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))   

# Pull latest code from git
if (PROJECT_ROOT / ".git").is_dir():
    print(f"Pulling latest code from {GIT_BRANCH}...")
    !git pull origin {GIT_BRANCH} || echo "git pull note: continuing with local files"

print(f"SpanCall project root: {PROJECT_ROOT}")

# Install package in editable mode along with optional dependencies
%pip install -q -e ".[model,data,agent,dev]" huggingface_hub

Cloning public repository from https://github.com/eleluong/spancall.git...
Cloning into 'spancall'...
remote: Enumerating objects: 206, done.
remote: Counting objects: 100% (206/206), done.
remote: Compressing objects: 100% (159/159), done.
remote: Total 206 (delta 51), reused 195 (delta 40), pack-reused 0 (from 0)
Receiving objects: 100% (206/206), 523.62 KiB | 2.10 MiB/s, done.
Resolving deltas: 100% (51/51), done.
Pulling latest code from main...
From https://github.com/eleluong/spancall
 * branch            main       -> FETCH_HEAD
Already up to date.
SpanCall project root: /content/spancall
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for spancall (pyproject.toml) ... done


### 0b. Configuration & Environment

Reads `.env` (or environment variables) for `DATASET_NAME` (`convoicon/spancall`), `HUGGING_FACE_TOKEN`, and LLM credentials (`BASE_URL`, `API_KEY`, `BASE_LLM_MODEL`), auto-discovers `PROJECT_ROOT` so `spancall` and `span_template` import directly, and configures device & paths.


In [2]:
import os, sys
from pathlib import Path

# Auto-discover project root (works whether cwd is repo root, notebooks/, or /content/spancall)
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "spancall" / "__init__.py").is_file():
    if (PROJECT_ROOT.parent / "spancall" / "__init__.py").is_file():
        PROJECT_ROOT = PROJECT_ROOT.parent
    elif (PROJECT_ROOT / "spancall" / "spancall" / "__init__.py").is_file():
        PROJECT_ROOT = PROJECT_ROOT / "spancall"
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from spancall.agent.backends.litellm_adapter import _load_dotenv_values

ENV = {**_load_dotenv_values(PROJECT_ROOT / ".env"), **os.environ}
HF_DATASET_ID = ENV.get("DATASET_NAME", "convoicon/spancall")
HF_TOKEN      = ENV.get("HUGGING_FACE_TOKEN") or ENV.get("HF_TOKEN")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

# --- optional Colab Drive persistence ---
DRIVE_MOUNT   = False
DRIVE_OUT     = "/content/drive/MyDrive/spancall_artifacts"

# --- data / checkpoints / eval ---
DATA_DIR      = str(PROJECT_ROOT / "data" / "processed")
TENSOR_DIR    = str(PROJECT_ROOT / "data" / "tensorized")
RESULTS_DIR   = str(PROJECT_ROOT / "results")
EVAL_SPLIT    = "hermes_eval"          # in-domain eval (TEST_ID)
TEST_SPLITS   = ["hermes_test_ood_schema", "hermes_test_ood_para", "hermes_test_adversarial"]

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import torch
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")

print(f"PROJECT_ROOT  = {PROJECT_ROOT}")
print(f"HF_DATASET_ID = {HF_DATASET_ID} (token={'set' if HF_TOKEN else 'unset'})")
print(f"DATA_DIR      = {DATA_DIR}")
print(f"TENSOR_DIR    = {TENSOR_DIR}")
print(f"DEVICE        = {DEVICE}")

PROJECT_ROOT  = /content/spancall
HF_DATASET_ID = convoicon/spancall (token=unset)
DATA_DIR      = /content/spancall/data/processed
TENSOR_DIR    = /content/spancall/data/tensorized
DEVICE        = cuda


---
## 1. Runtime & Package Check

Verify GPU/device availability and confirm `spancall` and `span_template` import cleanly.

In [3]:
!nvidia-smi --query-gpu=name,memory.total --format=csv 2>/dev/null || echo 'No NVIDIA GPU detected (using MPS / CPU fallback)'

name, memory.total [MiB]
NVIDIA L4, 23034 MiB


In [4]:
import spancall
import span_template
import transformers
import datasets
import huggingface_hub

print("spancall     :", spancall.__file__)
print("span_template:", span_template.__file__)
print("transformers :", transformers.__version__, "| datasets:", datasets.__version__)

spancall     : /content/spancall/spancall/__init__.py
span_template: /content/spancall/span_template/__init__.py
transformers : 5.22.0 | datasets: 4.8.5


### 1b. (Optional) Google Drive mount (Colab only)

Mounted artifacts under `DRIVE_OUT` keep results between Colab sessions.

In [5]:
if DRIVE_MOUNT:
    import shutil
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DRIVE_OUT, exist_ok=True)
    print('drive mounted:', DRIVE_OUT)

---
## 2. Dataset — Load via Hugging Face (`convoicon/spancall`)

Loads the 7 canonical SpanCall splits (`hermes_train`, `hermes_dev`, `hermes_calib`, `hermes_eval`, `hermes_test_ood_schema`, `hermes_test_ood_para`, `hermes_test_adversarial`) from `HF_DATASET_ID` (`convoicon/spancall`) on the Hugging Face Hub using `huggingface_hub.snapshot_download` (or `datasets.load_dataset`).

- If `DATA_DIR` already contains the `.jsonl` splits locally, they are used directly (set `FORCE_HF_DOWNLOAD = True` to re-pull from HF).
- You can also inspect any split directly as a Hugging Face `Dataset` via `datasets.load_dataset(HF_DATASET_ID, data_files=...)`.

In [6]:
from huggingface_hub import snapshot_download
from datasets import load_dataset

CANONICAL_SPLITS = [
    "hermes_train.jsonl",
    "hermes_dev.jsonl",
    "hermes_calib.jsonl",
    "hermes_eval.jsonl",
    "hermes_test_ood_schema.jsonl",
    "hermes_test_ood_para.jsonl",
    "hermes_test_adversarial.jsonl",
]

FORCE_HF_DOWNLOAD = False  # set True to always re-pull from Hugging Face Hub

os.makedirs(DATA_DIR, exist_ok=True)
missing = [f for f in CANONICAL_SPLITS if not os.path.exists(os.path.join(DATA_DIR, f))]

if FORCE_HF_DOWNLOAD or missing:
    print(f"Downloading splits from Hugging Face dataset: {HF_DATASET_ID} ...")
    snapshot_download(
        repo_id=HF_DATASET_ID,
        repo_type="dataset",
        local_dir=str(PROJECT_ROOT),
        allow_patterns=["data/processed/*.jsonl", "data/processed/*.json"],
        token=HF_TOKEN,
    )

print("Canonical splits in DATA_DIR:")
for f in sorted(os.listdir(DATA_DIR)):
    p = os.path.join(DATA_DIR, f)
    if os.path.isfile(p):
        print(f"  {f:<34} ({os.path.getsize(p):>12,} bytes)")

# Load via Hugging Face `datasets` to verify hub/local dataset integration
hf_ds = load_dataset(
    "json",
    data_files={
        "train": os.path.join(DATA_DIR, "hermes_train.jsonl"),
        "dev":   os.path.join(DATA_DIR, "hermes_dev.jsonl"),
        "calib": os.path.join(DATA_DIR, "hermes_calib.jsonl"),
        "eval":  os.path.join(DATA_DIR, "hermes_eval.jsonl"),
    },
)
print("\nHugging Face DatasetDict loaded:", hf_ds)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Canonical splits in DATA_DIR:
  hermes_calib.jsonl                 (   4,644,993 bytes)
  hermes_dev.jsonl                   (   4,624,634 bytes)
  hermes_eval.jsonl                  (  15,616,323 bytes)
  hermes_test_adversarial.jsonl      ( 129,893,828 bytes)
  hermes_test_ood_para.jsonl         (   5,785,351 bytes)
  hermes_test_ood_schema.jsonl       (  12,061,147 bytes)
  hermes_train.jsonl                 ( 102,226,436 bytes)
  prepare_report.json                (       3,543 bytes)


Generating train split: 0 examples [00:00, ? examples/s]

Generating dev split: 0 examples [00:00, ? examples/s]

Generating calib split: 0 examples [00:00, ? examples/s]

Generating eval split: 0 examples [00:00, ? examples/s]


Hugging Face DatasetDict loaded: DatasetDict({
    train: Dataset({
        features: ['sample_id', 'source_dataset', 'utterance', 'tools', 'ground_truth_calls', 'recent_context', 'current_time_iso', 'stratum', 'target_spans', 'split', 'metadata'],
        num_rows: 24071
    })
    dev: Dataset({
        features: ['sample_id', 'source_dataset', 'utterance', 'tools', 'ground_truth_calls', 'recent_context', 'current_time_iso', 'stratum', 'target_spans', 'split', 'metadata'],
        num_rows: 1720
    })
    calib: Dataset({
        features: ['sample_id', 'source_dataset', 'utterance', 'tools', 'ground_truth_calls', 'recent_context', 'current_time_iso', 'stratum', 'target_spans', 'split', 'metadata'],
        num_rows: 1718
    })
    eval: Dataset({
        features: ['sample_id', 'source_dataset', 'utterance', 'tools', 'ground_truth_calls', 'recent_context', 'current_time_iso', 'stratum', 'target_spans', 'split', 'metadata'],
        num_rows: 3426
    })
})


---
## 3. The Tensorizer — `UnifiedSample` → trainer rows (closes the P7 gap)

The trainer consumes **tensorized rows** (`t0/t1/t2/stage_c/dev/calib.jsonl`) with this exact per-row shape (see [`spancall/train/trainer.py`](../spancall/train/trainer.py) batch-format docstring):

```
input_ids, attention_mask   (L,) long
param_positions             (K,) long in [0, L)          # <|param|> anchor positions
utt_starts, utt_ends        (1,) long                    # utterance extent
cand_starts, cand_ends      (K, C) long, -1 padded       # candidate spans
pos_indices                 (K, P) long, -1 padded       # positive cand cols (0 = abstain)
reason_targets              (K,) long 0..4, -1 masked    # H4 reason classes
accept_targets              ()  float                    # stratum F → 1.0, A → 0.0
cue_mask                    (K,) float                   # pointer mask (all ones)
```

The cell below builds those rows per sample by:

1. Loading the tokenizer + marker tokens (same as `scripts/train.py`).
2. Rendering the **GT tool** with [`build_input`](../span_template/template.py) using the sample's `current_time_iso` as `now` (deterministic).
3. Tokenizing with `return_offsets_mapping` and **snapping each legend entry to the wholly-inside token range** (same `_entry_tok_range` logic the mask builder uses — cheaper than `tokenize_offset_mapping`, which is intentionally strict and can raise `bad_boundary` on real hub rows).
4. Building `CandidateMaskBuilder.build_param_masks` per param (P4) over the snapped offsets.
5. Mapping each `target_spans[param]` (char offsets / ctx_key) to its positive candidate column; `stratum A` params get the abstain slot (0); `h4_reasons` from `metadata`; cue_mask = ones.
6. Writing splits `t0/t1/t2/stage_c/dev/calib.jsonl` (graceful downsampling for fast runs).

> Row tensors are stored with `.tolist()` — `SpanCallTrainer.tensorize_rows` turns the JSON lists back into tensors.

In [7]:
import json, os, math
from datetime import date
from pathlib import Path
from typing import Any, Dict, List, Optional

import torch
from transformers import AutoTokenizer

from span_template import SPECIAL_MARKER_TOKENS, Entry, RecentRow, build_input
from spancall.data.align import tooldef_to_template_tool
from spancall.data.schema import Stratum, UnifiedSample
from spancall.model.masks import CandidateMaskBuilder

H4_REASONS = ("OK", "ABSENT", "AMBIGUOUS", "NEEDS_REASONING", "OFF_WINDOW")
STRATUM_ABSTAIN = {Stratum.A}                       # absent/unresolvable → abstain row

BACKBONE = "Qwen/Qwen3-Embedding-0.6B"
MAX_SEQ_LEN, MAX_SPAN_WIDTH = 4096, 16

tokenizer = AutoTokenizer.from_pretrained(BACKBONE, use_fast=True)
tokenizer.add_special_tokens({"additional_special_tokens": list(SPECIAL_MARKER_TOKENS)})
mask_builder = CandidateMaskBuilder(max_seq_len=MAX_SEQ_LEN, max_span_width=MAX_SPAN_WIDTH)
print('tokenizer ready; vocab =', len(tokenizer), '| MAX_SEQ_LEN =', MAX_SEQ_LEN)

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.71k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

tokenizer ready; vocab = 151684 | MAX_SEQ_LEN = 4096


In [8]:
from collections import Counter
import random, zlib
DROPS = Counter()                # drop-reason counter (reset per split)
DROP_UNALIGNED_F = True          # drop F rows whose GT spans cannot be aligned (else they'd be mislabelled "abstain")
SEED_DATA = 42

def _calib_half(sample_id) -> int:
    """Deterministic 50/50 split of hermes_calib: half 0 -> Stage C training, half 1 -> calibration (no leakage)."""
    return zlib.crc32(str(sample_id).encode()) % 2


def _now_iso(s: UnifiedSample) -> date:
    """Deterministic clock from the sample (R4): current_time_iso else fallback."""
    if s.current_time_iso:
        try:
            return date.fromisoformat(s.current_time_iso)
        except ValueError:
            pass
    return date(2026, 1, 1)


def _recent_rows(s: UnifiedSample) -> List[RecentRow]:
    """Build RecentRow list, dropping rows whose key/value trips R7 (reserved markers).

    The resolver / B1a baseline both treat such rows as abstain; here we keep the
    sample but drop only the infected recent rows so the rest still trains.
    """
    rows_out: List[RecentRow] = []
    for r in (s.recent_context or []):
        try:
            rows_out.append(RecentRow(key=str(r.get("key", "")), value=str(r.get("value", "")),
                                      source=str(r.get("source", "recent"))))
        except ValueError:
            continue   # R7: marker injection in key or value → drop this row only
    return rows_out


def _h4(s: UnifiedSample, param: str, default: str = "OK") -> int:
    reasons = (s.metadata.get("h4_reasons") or {}) if s.metadata else {}
    val = reasons.get(param, default)
    return H4_REASONS.index(val) if val in H4_REASONS else 0


MAX_POS = 3   # positive slots per param (pointer P in the v1 loss)


def _snap(cs: int, ce: int, offsets: List[tuple]) -> Optional[tuple]:
    """Lenient R8 snapping: first/last token wholly inside [cs, ce).

    Mirrors ``masks._entry_tok_range``; returns None when no token is inside.
    """
    first = last = None
    for i, (tcs, tce) in enumerate(offsets):
        if i >= MAX_SEQ_LEN:
            break
        if tcs >= cs and tce <= ce:
            if first is None:
                first = i
            last = i
    return (first, last) if first is not None else None


def _param_anchors(entries, offsets: List[tuple], n_params: int) -> List[int]:
    """Template-order ``<|param|>`` anchor token indexes (resolver §4.1.3, lenient)."""
    anchors: List[int] = []
    for e in entries:
        if getattr(e, "kind", None) != "param_anchor":
            continue
        r = _snap(e.char_start, e.char_end, offsets)
        anchors.append(r[0] if r is not None else 0)
    return (anchors + [0] * n_params)[:n_params]


def _sample_seq_len(s: UnifiedSample, tokenizer) -> int:
    """Compute tokenized sequence length of a sample (with fast char lower-bound)."""
    raw_chars = len(s.utterance or "") + sum(
        len(str(r.get("key", ""))) + len(str(r.get("value", ""))) + 16
        for r in (s.recent_context or [])
    )
    if raw_chars < (MAX_SEQ_LEN // 2):
        return raw_chars
    tool = s.tools[0] if s.tools else None
    if s.ground_truth_calls and s.tools:
        gt_name = s.ground_truth_calls[0].tool_name
        tool = next((t for t in s.tools if t.name == gt_name), tool)
    if tool is not None:
        try:
            out = build_input(
                tooldef_to_template_tool(tool),
                s.utterance,
                recent_context=_recent_rows(s),
                now=_now_iso(s),
            )
            return len(tokenizer(out.text, add_special_tokens=False)["input_ids"])
        except Exception:
            pass
    return len(tokenizer(s.utterance or "", add_special_tokens=False)["input_ids"])


def tensorize_sample(s: UnifiedSample, tokenizer, mask_builder) -> Optional[Dict[str, Any]]:
    """Build one trainer row from a UnifiedSample. Returns None on overflow/R7/empty-params."""
    if not s.tools or not s.ground_truth_calls:
        DROPS["no_tools_or_gt"] += 1
        return None
    gt_tool_name = s.ground_truth_calls[0].tool_name
    tool = next((t for t in s.tools if t.name == gt_tool_name), None)
    if tool is None or not tool.parameters:
        DROPS["gt_tool_missing_or_no_params"] += 1
        return None

    try:
        out = build_input(
            tooldef_to_template_tool(tool),
            s.utterance,
            recent_context=_recent_rows(s),
            now=_now_iso(s),
        )
    except Exception:
        DROPS["render_or_R7"] += 1
        return None

    enc = tokenizer(out.text, return_offsets_mapping=True, add_special_tokens=False)
    ids: List[int] = enc["input_ids"]
    offsets: List[tuple] = enc["offset_mapping"]
    if len(ids) > MAX_SEQ_LEN:
        DROPS["over_max_seq_len"] += 1
        return None

    # utterance extent from the parse (snapped to tokens)
    utt_entry = next((e for e in out.entries if getattr(e, "segment", None) == "utterance"), None)
    if utt_entry is None:
        DROPS["no_utterance_entry"] += 1
        return None
    usnap = _snap(utt_entry.char_start, utt_entry.char_end, offsets)
    if usnap is None:
        DROPS["utterance_snap_failed"] += 1
        return None
    utt_starts, utt_ends = usnap

    K = len(tool.parameters)                       # ToolDefinition.parameters, not .params
    anchors = _param_anchors(out.entries, offsets, K)

    cand_start_rows: List[List[int]] = []
    cand_end_rows: List[List[int]] = []
    pos_rows: List[List[int]] = []
    reasons: List[int] = []
    cue: List[float] = []
    align_fail = False   # a real (non-unresolved) GT target could not be matched to any candidate

    MAX_C = 256   # per-param candidate cap (trainer pads a batch to batch-max C)

    for idx, param in enumerate(tool.parameters):
        masks = mask_builder.build_param_masks(param, out.entries, offsets, out.text)
        cands = list(masks["candidates"])

        # ---- positives: ground-truth spans (char offsets or canonical value) ---
        targets = (s.target_spans or {}).get(param.name, [])
        matched_cis: List[int] = []
        for t in targets:
            if t.target_type == "unresolved":
                continue
            if t.target_type == "null_ctx":
                for ci, c in enumerate(cands):
                    if c.region == "null" and ci not in matched_cis:
                        matched_cis.append(ci)
                        break
                continue
            found = False
            if t.char_start is not None:
                for ci, c in enumerate(cands):
                    if c.char_start == t.char_start and c.char_end == t.char_end:
                        if ci not in matched_cis:
                            matched_cis.append(ci)
                        found = True
                        break
            if not found and t.canonical_value is not None:
                t_str = str(t.canonical_value).strip()
                for ci, c in enumerate(cands):
                    if c.canon_value is not None and (
                        c.canon_value == t.canonical_value or str(c.canon_value).strip() == t_str
                    ):
                        if ci not in matched_cis:
                            matched_cis.append(ci)
                        break

        if (s.stratum not in STRATUM_ABSTAIN and not matched_cis
                and any(t.target_type != "unresolved" for t in targets)):
            align_fail = True

        # Cap candidates to MAX_C while preserving any matched positive candidates
        # so pos_indices never exceed C (which would trigger a CUDA gather assert).
        if len(cands) > MAX_C:
            keep_pos = matched_cis[:MAX_POS]
            keep_set = set(keep_pos)
            other_cis = [ci for ci in range(len(cands)) if ci not in keep_set][: MAX_C - len(keep_pos)]
            kept_indices = sorted(keep_pos + other_cis)
            idx_map = {old_ci: new_ci for new_ci, old_ci in enumerate(kept_indices)}
            cands = [cands[ci] for ci in kept_indices]
            matched_cis = [idx_map[ci] for ci in keep_pos if ci in idx_map]

        cand_start_rows.append([c.tok_start for c in cands])
        cand_end_rows.append([c.tok_end for c in cands])

        pos_cols = [ci + 1 for ci in matched_cis[:MAX_POS]]  # 1-based into [abstain | A_k]
        if s.stratum in STRATUM_ABSTAIN:
            pos_cols = [0]                        # whole-param abstain
        if not pos_cols:
            pos_cols = [0]                        # safest default: abstain row

        pos_rows.append((pos_cols + [-1] * MAX_POS)[:MAX_POS])
        reasons.append(_h4(s, param.name))
        cue.append(1.0)

    if align_fail:
        DROPS["unaligned_positive"] += 1
        if DROP_UNALIGNED_F:
            return None   # avoid teaching "abstain" on rows that merely failed alignment

    # ---- pad candidates to a rectangular (K, C) grid (C >= 1) ----------------
    C = max((len(c) for c in cand_start_rows), default=0)
    C = max(min(C, MAX_C), 1)
    for i in range(len(cand_start_rows)):
        if len(cand_start_rows[i]) < C:
            cand_start_rows[i] += [-1] * (C - len(cand_start_rows[i]))
            cand_end_rows[i] += [-1] * (C - len(cand_end_rows[i]))
        else:
            cand_start_rows[i] = cand_start_rows[i][:C]
            cand_end_rows[i] = cand_end_rows[i][:C]

    row = {
        "input_ids": torch.tensor(ids, dtype=torch.long),
        "attention_mask": torch.ones(len(ids), dtype=torch.long),
        "param_positions": torch.tensor(anchors, dtype=torch.long),
        "utt_starts": torch.tensor([utt_starts], dtype=torch.long),
        "utt_ends": torch.tensor([utt_ends], dtype=torch.long),
        "cand_starts": torch.tensor(cand_start_rows, dtype=torch.long),
        "cand_ends": torch.tensor(cand_end_rows, dtype=torch.long),
        "pos_indices": torch.tensor(pos_rows, dtype=torch.long),
        "reason_targets": torch.tensor(reasons, dtype=torch.long),
        "accept_targets": torch.tensor(1.0 if s.stratum == Stratum.F else 0.0),
        "cue_mask": torch.tensor(cue, dtype=torch.float),
    }
    return row

In [9]:
# ---- filter out samples > 4K tokens & tensorize all splits --------------
import json, os, shutil
import multiprocessing as mp
from concurrent.futures import ThreadPoolExecutor
from contextlib import contextmanager
from typing import Tuple

# Disable internal Rust tokenizer thread pool so worker processes don't oversubscribe or deadlock
os.environ["TOKENIZERS_PARALLELISM"] = "false"
NUM_WORKERS = max(1, os.cpu_count() or 1)


def _serialize(row: Dict[str, Any]) -> Dict[str, Any]:
    return {k: (v.tolist() if hasattr(v, "tolist") else v) for k, v in row.items()}


def _check_line_seq_len(raw: str) -> Tuple[str, bool]:
    """Return (raw_line, keep). Short raw lines (< MAX_SEQ_LEN // 2 chars) cannot exceed MAX_SEQ_LEN tokens."""
    if len(raw) < (MAX_SEQ_LEN // 2):
        return raw, True
    s = UnifiedSample.from_dict(json.loads(raw))
    return raw, (_sample_seq_len(s, tokenizer) <= MAX_SEQ_LEN)


def _tensorize_raw_line(raw: str) -> Tuple[Optional[str], Optional[str]]:
    """Parse, tensorize, and JSON-serialize one line inside a worker. Returns (json_str, drop_reason)."""
    s = UnifiedSample.from_dict(json.loads(raw))
    DROPS.clear()
    row = tensorize_sample(s, tokenizer, mask_builder)
    if row is None:
        reason = next(iter(DROPS), "unknown")
        return None, reason
    return json.dumps(_serialize(row)), None


@contextmanager
def _parallel_map_runner(workers: int = NUM_WORKERS):
    """Yield a streaming parallel map function `pmap(fn, iterable, chunksize)` using fork workers (or threads fallback)."""
    if workers > 1 and "fork" in mp.get_all_start_methods():
        ctx = mp.get_context("fork")
        with ctx.Pool(processes=workers) as pool:
            yield lambda fn, it, chunksize=64: pool.imap(fn, it, chunksize=chunksize)
    elif workers > 1:
        with ThreadPoolExecutor(max_workers=workers) as ex:
            yield lambda fn, it, chunksize=64: ex.map(fn, it, chunksize=chunksize)
    else:
        yield lambda fn, it, chunksize=64: map(fn, it)


with _parallel_map_runner(NUM_WORKERS) as pmap:
    # 1. Remove any samples exceeding MAX_SEQ_LEN (4K) from canonical splits in DATA_DIR
    for split_file in CANONICAL_SPLITS:
        split_path = os.path.join(DATA_DIR, split_file)
        if not os.path.exists(split_path):
            continue
        with open(split_path, "r") as fh:
            raw_lines = [line.strip() for line in fh if line.strip()]
        # Only dispatch potentially long lines (>= MAX_SEQ_LEN // 2 chars) to workers
        long_indices = [i for i, r in enumerate(raw_lines) if len(r) >= (MAX_SEQ_LEN // 2)]
        if not long_indices:
            continue
        keep_flags = [True] * len(raw_lines)
        long_lines = [raw_lines[i] for i in long_indices]
        for idx, (_, keep) in zip(long_indices, pmap(_check_line_seq_len, long_lines, chunksize=64)):
            keep_flags[idx] = keep
        kept_lines = [r for r, k in zip(raw_lines, keep_flags) if k]
        removed_long = len(raw_lines) - len(kept_lines)
        if removed_long > 0:
            with open(split_path, "w") as fh:
                fh.write("\n".join(kept_lines) + "\n")
            print(f"filtered {split_file}: kept {len(kept_lines)}, removed {removed_long} (> {MAX_SEQ_LEN} tokens)")

    SPLIT_INPUTS = {
        "t0":       "hermes_train.jsonl",
        "t1":       "hermes_train.jsonl",
        "t2":       "hermes_train.jsonl",
        "stage_c":  "hermes_calib.jsonl",
        "dev":      "hermes_dev.jsonl",
        "calib":    "hermes_calib.jsonl",
    }

    # Subsampling limits for fast iteration (set to None for the full corpus)
    SAMPLE_LIMITS = {"t0": 500, "t1": None, "t2": None, "stage_c": None, "dev": None, "calib": None}  # T0 = small overfit sanity; rest = full data

    os.makedirs(TENSOR_DIR, exist_ok=True)

    # Cache raw lines per source file and reuse already-tensorized identical split specs (e.g. t1 == t2)
    _raw_file_cache: Dict[str, List[str]] = {}
    _completed_specs: Dict[ tuple, Tuple[str, Dict[str, Any]] ] = {}
    stats = {}

    for out_name, in_name in SPLIT_INPUTS.items():
        src = os.path.join(DATA_DIR, in_name)
        if not os.path.exists(src):
            print(f"SKIP {out_name}: {src} missing")
            continue
        limit = SAMPLE_LIMITS.get(out_name)
        half_filter = 0 if out_name == "stage_c" else (1 if out_name == "calib" else None)
        spec_key = (in_name, limit, half_filter)
        out_path = os.path.join(TENSOR_DIR, f"{out_name}.jsonl")

        # Reuse result when an identical split specification was already tensorized (e.g., t2 reuses t1)
        if spec_key in _completed_specs:
            prev_path, prev_stats = _completed_specs[spec_key]
            shutil.copyfile(prev_path, out_path)
            stats[out_name] = {
                "rows": prev_stats["rows"],
                "dropped": prev_stats["dropped"],
                "reasons": dict(prev_stats["reasons"]),
            }
            print(f"wrote {out_name}.jsonl (reused {os.path.basename(prev_path)}): "
                  f"{stats[out_name]['rows']} rows ({stats[out_name]['dropped']} dropped) "
                  f"reasons={stats[out_name]['reasons']}")
            continue

        if src not in _raw_file_cache:
            with open(src, "r") as fh:
                _raw_file_cache[src] = [line.strip() for line in fh if line.strip()]
        candidate_lines = _raw_file_cache[src]

        if half_filter is not None:     # Stage C and calibration use disjoint halves of hermes_calib
            candidate_lines = [
                raw for raw in candidate_lines
                if _calib_half(json.loads(raw).get("sample_id")) == half_filter
            ]
        if limit is not None:          # shuffle BEFORE slicing (files are not randomly ordered)
            candidate_lines = list(candidate_lines)
            random.Random(SEED_DATA).shuffle(candidate_lines)

        DROPS.clear()
        rows_json: List[str] = []
        dropped = 0
        if limit is not None:
            # Process in bounded slices so workers don't queue the entire file when limit is small
            batch_sz = max(limit * 3, 1024)
            for start in range(0, len(candidate_lines), batch_sz):
                batch = candidate_lines[start : start + batch_sz]
                for row_str, reason in list(pmap(_tensorize_raw_line, batch, chunksize=16)):
                    if row_str is None:
                        dropped += 1
                        DROPS[reason or "unknown"] += 1
                        continue
                    rows_json.append(row_str)
                    if len(rows_json) >= limit:
                        break
                if len(rows_json) >= limit:
                    break
        else:
            for row_str, reason in pmap(_tensorize_raw_line, candidate_lines, chunksize=64):
                if row_str is None:
                    dropped += 1
                    DROPS[reason or "unknown"] += 1
                    continue
                rows_json.append(row_str)

        with open(out_path, "w") as fh:
            if rows_json:
                fh.write("\n".join(rows_json) + "\n")
        stats[out_name] = {"rows": len(rows_json), "dropped": dropped, "reasons": dict(DROPS)}
        _completed_specs[spec_key] = (out_path, stats[out_name])
        print(f"wrote {out_name}.jsonl: {len(rows_json)} rows ({dropped} dropped) reasons={dict(DROPS)}")

print("\ntensorize summary:", json.dumps(stats, indent=2))

wrote t0.jsonl: 500 rows (653 dropped) reasons={'unaligned_positive': 561, 'gt_tool_missing_or_no_params': 65, 'render_or_R7': 27}
wrote t1.jsonl: 9560 rows (14511 dropped) reasons={'unaligned_positive': 12483, 'gt_tool_missing_or_no_params': 1503, 'render_or_R7': 525}
wrote t2.jsonl (reused t1.jsonl): 9560 rows (14511 dropped) reasons={'unaligned_positive': 12483, 'gt_tool_missing_or_no_params': 1503, 'render_or_R7': 525}
wrote stage_c.jsonl: 381 rows (475 dropped) reasons={'gt_tool_missing_or_no_params': 324, 'unaligned_positive': 83, 'render_or_R7': 68}
wrote dev.jsonl: 746 rows (974 dropped) reasons={'unaligned_positive': 207, 'render_or_R7': 124, 'gt_tool_missing_or_no_params': 643}
wrote calib.jsonl: 396 rows (466 dropped) reasons={'gt_tool_missing_or_no_params': 316, 'unaligned_positive': 93, 'render_or_R7': 57}

tensorize summary: {
  "t0": {
    "rows": 500,
    "dropped": 653,
    "reasons": {
      "unaligned_positive": 561,
      "gt_tool_missing_or_no_params": 65,
      "r

### Sanity: dev split tensorizes with computable positives

Quick diagnostic so you can see the ratio of fully-grounded rows before spending GPU time on training.

In [10]:
import json, os
dev_path = os.path.join(TENSOR_DIR, "dev.jsonl")
if os.path.exists(dev_path):
    with open(dev_path) as fh:
        rows = [json.loads(l) for l in fh if l.strip()]
    if not rows:
        print("dev.jsonl found but is empty (0 rows)")
    else:
        # A positive target is a candidate span index > 0 (0 is abstain, -1 is padding)
        n_hit = sum(
            1 for r in rows
            if any(pi > 0 for p_pos in r.get("pos_indices", []) for pi in p_pos)
        )
        print(f"dev.jsonl: {len(rows)} rows; rows with ≥1 positive target: {n_hit} ({n_hit/len(rows):.1%})")
        r = rows[0]
        print("sample row keys:", sorted(r.keys()))
        c_len = len(r["cand_starts"][0]) if r.get("cand_starts") and len(r["cand_starts"]) > 0 else 0
        print("input_ids:", len(r.get("input_ids", [])), "seq_len; params K =", len(r.get("param_positions", [])),
              "cand (K,C) =", len(r.get("cand_starts", [])), "x", c_len)
else:
    print("dev.jsonl not found — run the tensorizer cell first")

dev.jsonl: 746 rows; rows with ≥1 positive target: 309 (41.4%)
sample row keys: ['accept_targets', 'attention_mask', 'cand_ends', 'cand_starts', 'cue_mask', 'input_ids', 'param_positions', 'pos_indices', 'reason_targets', 'utt_ends', 'utt_starts']
input_ids: 335 seq_len; params K = 2 cand (K,C) = 2 x 256


---
## 4. Training — staged curriculum `T0 → T1 → T2 → C`

Runs [`scripts/train.py`](../scripts/train.py) against the tensorized splits.

- `--data_dir` → `TENSOR_DIR` above.
- `--config` → `configs/default.yaml` (backbone `Qwen/Qwen3-Embedding-0.6B`, heads H1/H3/H4/H8+accept, lr 3e-5/1e-4, bf16, batch 64, 4 epochs).
- **Fast / single-GPU overrides** are applied by generating a slim `configs/colab.yaml` (batch 2, 1 epoch, gradient accumulation 16, `data_dir` wired to `TENSOR_DIR`). Set `FULL_CONFIG = True` to use `default.yaml` unmodified.

**Embedding init — not a full re-init.** [`encoder.add_marker_tokens`](../spancall/model/encoder.py) only resizes the embedding table and initializes the **newly inserted marker-token rows** (`<|param|>`, …) with `mean + std` of the pretrained embeddings. The pretrained (Qwen3) embeddings are left untouched — no model is randomly re-initialized at the start of training.

**VRAM levers (all optional, default off).** Qwen3-Embedding-0.6B is ~0.6B params; a full fine-tune needs AdamW state (~3× the params) and 4K-token activations. The generated `configs/colab.yaml` enables:

- **Gradient checkpointing** (`training.gradient_checkpointing: true`) — recomputes activations in backward, cutting ~40% of activation memory.
- **Frozen backbone for T0/T1** (`training.freeze_encoder_stages: [T0, T1]`) — sanity/pointer stages train heads only (`H1/H3/H4/H8`), so no encoder grads and no AdamW state for the backbone during those stages.
- **Mixed precision** — bf16 by default (no scaler needed). On T4-class GPUs emulated bf16 is slow: set `mixed_precision: fp16` for the same memory at hardware speed (automatic `GradScaler`).
- **8-bit optimizer** (`training.use_8bit_optimizer: true`, optional) — `bitsandbytes` `PagedAdamW8bit`, cut optimizer state ~75%; falls back to AdamW if `bitsandbytes` isn't installed.

In [11]:
import yaml, os

FULL_CONFIG = False                 # True → use configs/default.yaml untouched
EXPERIMENT_NAME = "spancall_colab"
SEED = 42

if not FULL_CONFIG:
    with open("configs/default.yaml") as fh:
        cfg = yaml.safe_load(fh)
    cfg["experiment"]["name"] = EXPERIMENT_NAME
    cfg["experiment"]["seed"] = SEED
    cfg["experiment"]["output_dir"] = RESULTS_DIR
    cfg["experiment"]["data_dir"] = TENSOR_DIR
    # L4 (~23GB VRAM): 16-bit (bf16), batch 8, grad-accum 4 (effective 32), length-bucketed + token-capped batches
    cfg["optimization"] = {**cfg.get("optimization", {}), "batch_size": 8, "num_epochs": 3}
    cfg["training"] = {**cfg.get("training", {}), "gradient_accumulation_steps": 4}
    # ---- 16-bit & VRAM levers -------------------------------------------
    # 16-bit bf16 (backbone + autocast); swap to "fp16" on older T4 GPUs if needed.
    cfg["optimization"]["mixed_precision"] = "bf16"   # 16-bit ("bf16" or "fp16")
    cfg["training"]["gradient_checkpointing"] = True    # Required for 28-layer Qwen3-0.6B at 4K seq_len (~15x less activation VRAM)
    cfg["training"]["freeze_encoder_stages"] = []       # Train encoder + marker tokens in T0/T1 so E1 pointer gate (>=0.80) passes
    # cfg["training"]["use_8bit_optimizer"] = True   # optional PagedAdamW8bit (needs bitsandbytes)
    cfg["training"]["stop_on_gate_kill"] = False   # True -> abort after T1 when E1 gate says KILL (<0.80)
    cfg["model"]["max_seq_len"] = MAX_SEQ_LEN
    out = "configs/colab.yaml"
    with open(out, "w") as fh:
        yaml.safe_dump(cfg, fh)
    print("wrote", out)
else:
    out = "configs/default.yaml"
print("config =", out)
CONFIG_PATH = out

wrote configs/colab.yaml
config = configs/colab.yaml


In [12]:
# Direct in-notebook training execution (provides live logging and full native tracebacks)
import gc, torch
from scripts.train import main as train_main

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

train_main([
    "--config", CONFIG_PATH,
    "--experiment_name", EXPERIMENT_NAME,
    "--output_dir", RESULTS_DIR,
    "--data_dir", TENSOR_DIR,
    "--stages", "T0,T1,T2,C",
    "--device", DEVICE,
    "--seed", str(SEED),
])

[SpanCall Train] config: configs/colab.yaml
[SpanCall Train] backbone=Qwen/Qwen3-Embedding-0.6B heads=('tool', 'pointer', 'abstain_reason', 'correctness') pack_tools=1


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.19GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


[train:T0] start: rows=500 epochs=3 steps/epoch=32 total_steps=96 bs=16


... (4 outputs left out)

eval[dev]:   0%|          | 0/47 [00:00<?, ?it/s]

[eval:dev] exact_payload_match=0.7274 n=1416 | span_acc=0.6827 (n=520) abstain_acc=0.7533 (n=896) pred_abstain_rate=0.525
[train:T1] GATE KILL: exact_payload_match=0.7274 < 0.80 (E1 pointer ceiling not met)
[train:T1] finished: {'loss': 1.0225380389985432, 'steps': 1794.0, 'status': 'ok', 'n': 1416.0, 'exact_payload_match': 0.7274011299435028, 'span_acc': 0.6826923076923077, 'n_span': 520.0, 'abstain_acc': 0.7533482142857143, 'n_gold_abstain': 896.0, 'pred_abstain_rate': 0.5247175141242938, 'gate_proceed': False, 'gate_kill': True}
[train:T2] start: rows=9560 epochs=3 steps/epoch=598 total_steps=1794 bs=16


train[T2]:   0%|          | 0/1794 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 70.00 MiB. GPU 0 has a total capacity of 22.03 GiB of which 13.12 MiB is free. Including non-PyTorch memory, this process has 22.01 GiB memory in use. Of the allocated memory 20.17 GiB is allocated by PyTorch, and 1.61 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
# T1 pointer-ceiling gate check (E1: >=0.90 proceed, <0.80 kill). Downstream results are meaningless if killed.
import glob, json
STOP_ON_GATE_KILL = False   # True -> raise so later cells don't run on a failed model
_exp = os.path.join(RESULTS_DIR, f"{EXPERIMENT_NAME}_seed{SEED}")
_t1 = {}
for _mp in sorted(glob.glob(os.path.join(_exp, "*", "checkpoints", "metrics.json"))):
    try:
        _m = json.load(open(_mp))
    except Exception:
        continue
    if isinstance(_m.get("T1"), dict):
        _t1 = _m["T1"]
print("T1 summary:", _t1)
if _t1.get("gate_kill"):
    msg = f"!!! T1 gate KILL (exact_payload_match={_t1.get('exact_payload_match')}) — fix labels/alignment before trusting T2/C or any eval !!!"
    print("=" * 72); print(msg); print("=" * 72)
    if STOP_ON_GATE_KILL:
        raise RuntimeError(msg)


### Pick the best checkpoint

The trainer writes checkpoints per stage under `results/{exp}_seed{seed}/{stage}/checkpoints/`. We pick:

1. The **stage-C checkpoint** if it exists (full curriculum, frozen-encoder H8), else
2. the **last stage with a checkpoint** (T2 → T1 → T0), else
3. the `model_init` dir saved by `scripts/train.py`.

`CHECKPOINT_DIR` is then used by prediction/calibration/eval.

In [ ]:
import glob, os

exp_root = os.path.join(RESULTS_DIR, f"{EXPERIMENT_NAME}_seed{SEED}")

def _latest_stage_cp():
    for stage in ["C", "T2", "T1", "T0"]:
        cps = sorted(glob.glob(os.path.join(exp_root, stage, "checkpoints", "**", "model.pt"), recursive=True))
        if cps:
            return os.path.dirname(cps[-1]), stage
    return None, None

CHECKPOINT_DIR = None
cp_dir, cp_stage = _latest_stage_cp()
model_init = os.path.join(RESULTS_DIR, "model_init")

if cp_dir is not None:
    CHECKPOINT_DIR = cp_dir
    print(f"picked stage-{cp_stage} checkpoint @ {CHECKPOINT_DIR}")
elif os.path.isdir(model_init):
    # model_init is saved by scripts/train.py BEFORE any training step
    # (pretrained backbone + randomly-initialized heads + marker rows).
    # Using it means inference runs on an UNTRAINED model — warn loudly.
    CHECKPOINT_DIR = model_init
    print("=" * 72)
    print("!!! WARNING: no staged checkpoint found — using model_init (UNTRAINED model) !!!")
    print("    scripts/train.py saves model_init before training; only the marker-token")
    print("    rows + heads were initialized. Re-run section 4 (or a later stage) first.")
    print("=" * 72)
else:
    all_m = sorted(glob.glob(os.path.join(RESULTS_DIR, "**", "model.pt"), recursive=True))
    if all_m:
        CHECKPOINT_DIR = os.path.dirname(all_m[-1])
        print("fallback checkpoint @", CHECKPOINT_DIR)

print("CHECKPOINT_DIR =", CHECKPOINT_DIR)
if CHECKPOINT_DIR:
    print("  contains:", sorted(os.listdir(CHECKPOINT_DIR)))

---
## 5. Inference — generate model predictions with `SpanCallResolver`

Loads the chosen checkpoint into [`SpanCallResolver`](../spancall/infer/resolver.py) (same encoder + marker-token setup as training, so `load_state_dict` lines up) and emits prediction rows over a split.

Prediction row contract (consumed by `scripts/eval_toolcall.py`):
```
{sample_id, hit, payload, tool_name, accept_probability}
```

In [ ]:
import json, os
from typing import Any, Dict, List, Optional

import torch
from transformers import AutoTokenizer

from span_template import SPECIAL_MARKER_TOKENS
from spancall.data.schema import UnifiedSample
from spancall.infer.resolver import SpanCallResolver
from spancall.model.encoder import SpanCallTransformerEncoder
from spancall.model.model import SpanCallModel
from spancall.train.config import TrainingConfig

cfg = TrainingConfig.from_yaml(CONFIG_PATH)
tokenizer = AutoTokenizer.from_pretrained(cfg.model.backbone_name, use_fast=True)
_enc_kw = {"dtype": torch.bfloat16} if (DEVICE == "cuda" and cfg.optimization.mixed_precision == "bf16") else {}
encoder = SpanCallTransformerEncoder(
    model_name_or_path=cfg.model.backbone_name,
    hidden_dropout_prob=cfg.model.dropout,
    tokenizer=tokenizer,
    **_enc_kw,
)
encoder.add_marker_tokens(list(SPECIAL_MARKER_TOKENS))
model = SpanCallModel(
    encoder=encoder,
    heads=cfg.model.heads,
    head_dim=cfg.model.head_dim,
    max_span_width=cfg.model.max_span_width,
    hidden_dim=encoder.hidden_dim,
)

# calib_thresholds_path → calibration_report.json from §6 (optional)
calib_path = None
if CHECKPOINT_DIR and os.path.exists(os.path.join(CHECKPOINT_DIR, "calibration_report.json")):
    calib_path = os.path.join(CHECKPOINT_DIR, "calibration_report.json")

resolver = SpanCallResolver(
    model_path=CHECKPOINT_DIR,
    calib_thresholds_path=calib_path,
    encoder=encoder,
    model=model,
    tokenizer=tokenizer,
    max_seq_len=cfg.model.max_seq_len,
    max_span_width=cfg.model.max_span_width,
)
if DEVICE in ("cuda", "mps"):
    model.to(DEVICE)
print("resolver ready (checkpoint =", CHECKPOINT_DIR, ")")

In [ ]:
# Diagnostics: (1) did the checkpoint weights actually land in `model`? (2) what does the resolver do on a few rows?
import random
ck = torch.load(os.path.join(CHECKPOINT_DIR, "model.pt"), map_location="cpu")
sd = ck if isinstance(ck, dict) and any(hasattr(v, "shape") for v in ck.values()) else (
    ck.get("model_state_dict") or ck.get("model") or ck.get("state_dict") or ck)
cur = model.state_dict()
common = [k for k in cur if k in sd]
n_diff = sum(1 for k in common if not torch.allclose(cur[k].cpu().float(), sd[k].float()))
print(f"model keys={len(cur)} | in checkpoint={len(common)} | differing from checkpoint={n_diff}")
if len(common) < len(cur) or n_diff > 0:
    print("!!! checkpoint NOT (fully) loaded into the resolver model — predictions are from the wrong weights !!!")
else:
    print("checkpoint weights match the live model.")

_src = os.path.join(DATA_DIR, f"{EVAL_SPLIT}.jsonl")
_lines = [l for l in open(_src) if l.strip()]
for l in random.Random(0).sample(_lines, 5):
    s = UnifiedSample.from_dict(json.loads(l))
    res = resolver.try_resolve_candidates(s.tools, s.utterance, s.recent_context, s.current_time_iso)
    gt = s.ground_truth_calls[0] if s.ground_truth_calls else None
    print("-" * 60)
    print("utt     :", (s.utterance or "")[:120])
    print("stratum :", s.stratum, "| GT tool:", gt.tool_name if gt else None, "| GT args:", gt.arguments if gt else None)
    print("pred    : hit=%s tool=%s accept=%.3f payload=%s" % (res.hit, res.tool_name, float(res.accept_score), res.payload))


In [ ]:
EVAL_LIMIT, TEST_LIMIT = 1000, 500   # same random subset is used for the model AND the B1a baseline

def predict_split(split_name: str, out_path: str, limit: Optional[int] = None, seed: int = 42) -> None:
    src = os.path.join(DATA_DIR, f"{split_name}.jsonl")
    if not os.path.exists(src):
        print(f"no source {src} — skipped")
        return
    import random
    with open(src) as fh:
        lines = [l.strip() for l in fh if l.strip()]
    if limit is not None and len(lines) > limit:
        lines = random.Random(seed).sample(lines, limit)
    os.makedirs(RESULTS_DIR, exist_ok=True)
    subset_path = os.path.join(RESULTS_DIR, f"subset_{split_name}.jsonl")
    with open(subset_path, "w") as fh:               # evaluators read this subset (apples-to-apples)
        for l in lines:
            fh.write(l + "\n")
    preds: List[Dict[str, Any]] = []
    for line in lines:
        s = UnifiedSample.from_dict(json.loads(line))
        res = resolver.try_resolve_candidates(s.tools, s.utterance, s.recent_context, s.current_time_iso)
        preds.append({
            "sample_id": s.sample_id,
            "hit": bool(res.hit),
            "payload": res.payload,
            "tool_name": res.tool_name,
            "accept_probability": float(res.accept_score),
        })
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with open(out_path, "w") as fh:
        for p in preds:
            fh.write(json.dumps(p) + "\n")
    print(f"predicted {len(preds)} rows (hit={sum(p['hit'] for p in preds)}) → {out_path}")

# In-domain eval split
predict_split(EVAL_SPLIT, os.path.join(RESULTS_DIR, "preds_eval.jsonl"), limit=EVAL_LIMIT)


---
## 6. Calibration (optional)

Produces `calibration_report.json` (temperature + per-tier thresholds) which the resolver can then consume via `calib_thresholds_path`.

The calibrator expects **resolver prediction rows** with this contract (see [`spancall/train/calibrate.py`](../spancall/train/calibrate.py)):
```
{sample_id, tool_name, payload, param_probabilities, joint_margin, accept_probability, schema_valid, risk_tier}
+ parallel ground truth {sample_id, arguments}
```

> If you prefer to skip temperature fitting, pass `--temperature 1.0` and the report still sweeps tau_p/tau_a/tau_delta.

In [ ]:
import json, os

# Build calibration rows from the calib split via the resolver
CALIB_DATA = os.path.join(RESULTS_DIR, "calib_rows.jsonl")
CALIB_GT   = os.path.join(RESULTS_DIR, "calib_gt.jsonl")

src = os.path.join(DATA_DIR, "hermes_calib.jsonl")
preds, gts = [], []
if os.path.exists(src):
    with open(src) as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            s = UnifiedSample.from_dict(json.loads(line))
            if _calib_half(s.sample_id) != 1:   # half 0 was used for Stage C training
                continue
            res = resolver.try_resolve_candidates(s.tools, s.utterance, s.recent_context, s.current_time_iso)

            # per-param probabilities (only available for resolved params)
            param_probs = {p.name: float(p.confidence) for p in res.parameters.values()}
            preds.append({
                "sample_id": s.sample_id,
                "tool_name": res.tool_name,
                "payload": res.payload,
                "param_probabilities": param_probs,
                "joint_margin": float(res.joint_margin),
                "accept_probability": float(res.accept_score),
                "schema_valid": bool(res.payload is not None),
                "risk_tier": next((t.risk_tier for t in s.tools if t.name == res.tool_name),
                                    s.tools[0].risk_tier if s.tools else "read_only"),
            })
            gt_args = s.ground_truth_calls[0].arguments if s.ground_truth_calls else {}
            gts.append({"sample_id": s.sample_id, "arguments": gt_args})

with open(CALIB_DATA, "w") as fh:
    for p in preds:
        fh.write(json.dumps(p) + "\n")
with open(CALIB_GT, "w") as fh:
    for g in gts:
        fh.write(json.dumps(g) + "\n")
print(f"wrote {len(preds)} calib rows → {CALIB_DATA}")

In [ ]:
CALIB_REPORT = os.path.join(CHECKPOINT_DIR or RESULTS_DIR, "calibration_report.json")

import subprocess, sys
subprocess.run([
    sys.executable, "scripts/calibrate.py",
    "--config", CONFIG_PATH,
    "--checkpoint", CHECKPOINT_DIR,
    "--calib_data", CALIB_DATA,
    "--ground_truth", CALIB_GT,
    "--output", CALIB_REPORT,
], check=False)

if os.path.exists(CALIB_REPORT):
    import json
    rep = json.load(open(CALIB_REPORT))
    print("report keys:", sorted(rep.keys()))
    print("temperature:", rep.get("temperature"), "| tau_p:", rep.get("tau_p"),
          "| tau_a:", rep.get("tau_a"), "| tau_delta:", rep.get("tau_delta"))

In [ ]:
# Rebuild the resolver WITH the calibration thresholds, then re-predict the eval subset (gated).
if os.path.exists(CALIB_REPORT):
    resolver = SpanCallResolver(
        model_path=CHECKPOINT_DIR,
        calib_thresholds_path=CALIB_REPORT,
        encoder=encoder, model=model, tokenizer=tokenizer,
        max_seq_len=cfg.model.max_seq_len, max_span_width=cfg.model.max_span_width,
    )
    if DEVICE in ("cuda", "mps"):
        model.to(DEVICE)
    predict_split(EVAL_SPLIT, os.path.join(RESULTS_DIR, "preds_eval.jsonl"), limit=EVAL_LIMIT)
else:
    print("no calibration report — eval predictions stay ungated")


---
## 7. Offline evaluation

Two runs of [`scripts/eval_toolcall.py`](../scripts/eval_toolcall.py):

1. **B1a baseline** — deterministic regex extraction (`--baseline b1a`).
2. **SpanCall model** — `--predictions preds_eval.jsonl` generated in §5.

If you generate predictions with `calib_thresholds_path` set in §5, the whole-call gate (tau_p/tau_a/tau_delta) is applied at prediction time and `hit` reflects gated agreement.

In [ ]:
import subprocess, sys
subprocess.run([
    sys.executable, "scripts/eval_toolcall.py",
    "--test_data", f"{RESULTS_DIR}/subset_{EVAL_SPLIT}.jsonl",
    "--baseline", "b1a",
    "--output", f"{RESULTS_DIR}/toolcall_metrics_{EVAL_SPLIT}_b1a.json",
], check=True)

In [ ]:
import subprocess, sys
subprocess.run([
    sys.executable, "scripts/eval_toolcall.py",
    "--test_data", f"{RESULTS_DIR}/subset_{EVAL_SPLIT}.jsonl",
    "--predictions", f"{RESULTS_DIR}/preds_eval.jsonl",
    "--output", f"{RESULTS_DIR}/toolcall_metrics_{EVAL_SPLIT}_model.json",
], check=True)

### Compare baselines

Load both metric files and print the headline table.

In [ ]:
import json, os

def _load_metrics(path):
    if not os.path.exists(path):
        return None
    with open(path) as fh:
        return json.load(fh)

rows = []
for label, path in [
    ("B1a (regex)", os.path.join(RESULTS_DIR, f"toolcall_metrics_{EVAL_SPLIT}_b1a.json")),
    ("SpanCall",    os.path.join(RESULTS_DIR, f"toolcall_metrics_{EVAL_SPLIT}_model.json")),
]:
    m = _load_metrics(path)
    if m is None:
        continue
    rows.append({
        "system": label,
        "n": m.get("total_samples"),
        "exact_payload_match": round(m.get("exact_payload_match", 0.0), 3),
        "hit_rate": round(m.get("hit_rate", 0.0), 3),
        "precision_at_hit": round(m.get("precision_at_hit", 0.0), 3),
        "false_hit_on_A": round(m.get("false_hit_rate_on_a", 0.0), 4),
        "ECE": round(m.get("ece", 0.0), 4),
    })

if rows:
    headers = ["system", "n", "exact_payload_match", "hit_rate", "precision_at_hit", "false_hit_on_A", "ECE"]
    print(" | ".join(headers))
    print("-|" * len(headers))
    for r in rows:
        print(" | ".join(str(r[h]) for h in headers))
else:
    print("no metric files found yet — run the two eval_toolcall cells above")

---
## 8. Test — OOD / adversarial splits

Evaluate the trained checkpoint (or the B1a baseline) on the three held-out test splits:

- `hermes_test_ood_schema` — schema-disjoint OOD tools
- `hermes_test_ood_para` — paraphrase OOD
- `hermes_test_adversarial` — adversarial/ambiguity injection

> These are **unseen** at training time (partitioned by schema-disjoint split in `prepare_data.py`), so they measure generalization, not memorization.

In [ ]:
# generate predictions for each test split (limit keeps runtime sane)
for split in TEST_SPLITS:
    predict_split(split, os.path.join(RESULTS_DIR, f"preds_{split}.jsonl"), limit=TEST_LIMIT)

In [ ]:
import subprocess, sys

def _run_eval(split: str, label: str, extra: list) -> None:
    out = f"{RESULTS_DIR}/toolcall_metrics_{split}_{label}.json"
    cmd = [
        sys.executable, "scripts/eval_toolcall.py",
        "--test_data", f"{RESULTS_DIR}/subset_{split}.jsonl",
        *extra,
        "--output", out,
    ]
    subprocess.run(cmd, check=False)
    print(f"eval {split} [{label}] -> {out}")

for split in TEST_SPLITS:
    _run_eval(split, "b1a", ["--baseline", "b1a"])
    _run_eval(split, "model", ["--predictions", f"{RESULTS_DIR}/preds_{split}.jsonl"])

print("test eval done")

In [ ]:
import json, os

print(f"{'split':<26}{'system':<8}{'n':>6}{'exact':>8}{'hit':>7}{'P@hit':>8}{'falseA':>9}{'ECE':>8}")
for split in TEST_SPLITS + [EVAL_SPLIT]:
    for label, suffix in [("b1a", "b1a"), ("model", "model")]:
        path = os.path.join(RESULTS_DIR, f"toolcall_metrics_{split}_{suffix}.json")
        m = _load_metrics(path)
        if not m:
            continue
        n = m.get("total_samples", "-")
        print(f"{split:<26}{label:<8}{n:<6}"
              f"{m.get('exact_payload_match', 0.0):<8.3f}{m.get('hit_rate', 0.0):<7.3f}"
              f"{m.get('precision_at_hit', 0.0):<8.3f}{m.get('false_hit_rate_on_a', 0.0):<9.4f}"
              f"{m.get('ece', 0.0):<8.4f}")

---
## 9. Persist to Drive (optional)

Mirror the artifacts (tensorized data, checkpoints, metrics) to the mounted Drive folder so nothing is lost when a Colab VM terminates.

In [ ]:
if DRIVE_MOUNT:
    import shutil, os
    os.makedirs(DRIVE_OUT, exist_ok=True)
    for name, src in [
        ("data_tensorized", TENSOR_DIR),
        ("results", RESULTS_DIR),
        ("data_processed", DATA_DIR),
    ]:
        if os.path.isdir(src):
            dst = os.path.join(DRIVE_OUT, name)
            shutil.copytree(src, dst, dirs_exist_ok=True)
            print(f"mirrored {src} → {dst}")
    print("\nartifacts persisted to", DRIVE_OUT)

---
## 9b. Long-trajectory agent check (context plumbing)

Runs `scripts/eval_agent.py --benchmark long_trajectory` at 8k/12k/16k context budgets with the checkpoint-free scripted resolver (validates trajectory → `recent_context` feeding, per-turn stats, and LLM-path trimming). With a trained `CHECKPOINT_DIR`, drop `--scripted_resolver` and pass `--spancall_checkpoint` to measure real hit-rate (needs calibration first).

In [ ]:
import subprocess, sys, json, os
for ctx, budget in [(8192, 4096), (12288, 6144), (16384, 8192)]:
    out = f"{RESULTS_DIR}/long_{ctx // 1024}k_nb.json"
    subprocess.run([
        sys.executable, "scripts/eval_agent.py", "--benchmark", "long_trajectory",
        "--scripted_resolver", "--max_context_tokens", str(ctx),
        "--context_budget_tokens", str(budget), "--output", out,
    ], check=False)
    if os.path.exists(out):
        r = json.load(open(out))
        print(ctx, {k: v.get("spancall_hits") for k, v in r.get("per_turn_stats", {}).items()})

---
## 10. Test suite (repo unit tests)

Runs the repo's own tests to confirm the checked-in code is healthy in this environment (template round-trip, masks, heads/loss shapes, gate, calibration).

In [ ]:
!python -m pytest tests -q --disable-warnings --no-header -p no:cacheprovider 2>&1 | tail -25

In [ ]:
print(f"""
== SpanCall pipeline complete ==

What ran:
  0. git clone/pull + editable pip install
  1. env setup (.env loaded) + direct imports check
  2. HF dataset ({HF_DATASET_ID}) → {DATA_DIR}
  3. tensorizer → {TENSOR_DIR} (closes P7 gap)
  4. staged training T0/T1/T2/C → {RESULTS_DIR}
  5. resolver predictions → {RESULTS_DIR}/preds_*.jsonl
  6. (optional) calibration → calibration_report.json
  7. offline eval: b1a + model on {EVAL_SPLIT}
  8. test splits: {TEST_SPLITS}
  9. optional Drive mirror @ {DRIVE_OUT}
  9b. long-trajectory agent check
 10. repo unit tests
""")